# 06 — YOLOv8 Training v2 (Resolution Scaling + Copy-Paste Augmentation)

- **Resolution**: 1280px (native BDD100K resolution, up from 640px)
- **Copy-Paste Augmentation**: copy_paste=0.3 to address minority class imbalance

In [ ]:
from pathlib import Path

_here = Path.cwd().resolve()
for _root in (_here, *_here.parents):
    if (_root / "src").is_dir() and (_root / "outputs").is_dir():
        break
else:
    raise FileNotFoundError("Repo root not found (need src/ and outputs/).")

print(f"Repo root: {_root}")

In [ ]:
!pip install ultralytics --no-deps

In [ ]:
import sys
import os
from pathlib import Path
import random
import numpy as np

_here = Path.cwd().resolve()
for _root in (_here, *_here.parents):
    if (_root / "src").is_dir() and (_root / "outputs").is_dir():
        break
else:
    raise FileNotFoundError("Repo root not found (need src/ and outputs/).")

if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.utils import seed_everything, log_environment

SEED = 42
seed_everything(SEED)
log_environment()

In [ ]:
import os

DATA_CONFIG = _root / "configs" / "yolov8_bdd100k_v2.yaml"
PROJECT_DIR = _root / "outputs" / "bdd100k_project" / "runs"

os.makedirs(PROJECT_DIR, exist_ok=True)

print(f"Data config: {DATA_CONFIG}")
print(f"Project dir: {PROJECT_DIR}")

## Phase 5 Augmentation Settings

**copy_paste=0.3** added to address minority class imbalance.
This augmentation pastes object instances from other images, effectively upsampling rare classes.

In [ ]:
AUGMENTATION = dict(
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.0,
    translate=0.1,
    scale=0.5,
    degrees=0.0,
    copy_paste=0.3,
)

## Full Training — YOLOv8m v2 (50 epochs)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")

results = model.train(
    data=str(DATA_CONFIG),
    epochs=50,
    imgsz=1280, 
    batch=8,
    name="yolov8m_bdd100k_v2",
    project=str(PROJECT_DIR),
    device=0,
    seed=SEED,
    patience=15,
    save=True,
    plots=True,
    workers=8,
    **AUGMENTATION,
)